In [2]:
# Exercise 1 — check your environment: Python, the Week 10 and 11 packages, Ollama and two models.
# Ollama = the free app that runs the AI models on your own laptop. Each check prints one line:
# OK, or NOT OK followed by the fix to try. Result: the settings below plus a list of checks.
# PREPARED: just run it. No Ollama (or a slow laptop)? Set USE_RECORDED = True to replay the recorded real run.
import base64, hashlib, importlib, json, platform, re, sys, time   # standard-library helpers used in this notebook
from pathlib import Path                                  # file paths for Windows and macOS

# Settings used by the whole notebook. Replay mode = reuse the saved real replies of a finished run,
# so you do not need Ollama. Live mode = really ask the models on your laptop.
USE_RECORDED = False                       # True = replay mode: use the recorded real replies and vectors (no Ollama). False = live: ask Ollama.
# 127.0.0.1 means "this computer". We avoid "localhost": on Windows it adds about 2 s per request.
OLLAMA_URL = "http://127.0.0.1:11434"      # 127.0.0.1, not localhost: on Windows localhost adds ~2 s per request
MODEL = "qwen2.5:3b"                       # the chat model, as in Week 10 (qwen2.5:1.5b works, but replay will not match)
# An embedding = a list of numbers (a vector) that captures the meaning of a text.
EMBED_MODEL = "nomic-embed-text:latest"    # the embedding model: turns a text into 768 numbers
SEED = 11523                               # fixed seed: usually the same reply on the same machine (not guaranteed; see Step 0.5)

# Helper: print one line per check, so you can see at a glance what works and what to fix.
def check(name, ok, fix):                                 # print one OK / NOT OK line, with the fix if needed
    print(("OK      " if ok else "NOT OK  ") + name + ("" if ok else "   ->  FIX: " + fix))   # one line per check

# Check 1: which Python runs this notebook, and is it new enough?
print("Python", sys.version.split()[0], "on", platform.system())   # which Python runs this notebook
check("Python 3.9 or newer", sys.version_info >= (3, 9), "install Python 3.11 from python.org")   # minimum version
# Check 2: can each Week 10 package be imported (import = load a package so that we can use it)?
for pkg in ["pandas", "requests", "sklearn", "matplotlib", "pydantic"]:   # the Week 10 packages
    try:                                                  # try to import it
        importlib.import_module(pkg)                      # import the package from its name
        check("package " + pkg + " (Week 10)", True, "")  # it is installed
    except ImportError:                                   # the import failed: it is not installed
        check("package " + pkg + " (Week 10)", False, "do Week 10 Step 0 first, then restart the kernel")   # fix
# Check 3: the same test for the two new Week 11 packages.
for pkg in ["numpy", "chromadb"]:                         # the Week 11 packages (numpy comes with pandas)
    try:                                                  # try to import it
        importlib.import_module(pkg)                      # import the package from its name
        check("package " + pkg + " (Week 11)", True, "")  # it is installed
    except ImportError:                                   # the import failed: it is not installed
        check("package " + pkg + " (Week 11)", False, "pip install -r requirements_week11.txt, then restart the kernel")   # fix

# Check 4: is Ollama running, and are both models downloaded? (Skipped in replay mode.)
import requests                                           # HTTP library used to talk to Ollama
if USE_RECORDED:                                          # replay mode: Ollama is not needed
    print("SKIPPED (replay mode)  Ollama server, " + MODEL + ", " + EMBED_MODEL)   # nothing to check
else:                                                     # live mode: Ollama must be running
    try:                                                  # ask Ollama which models are downloaded
        # The /api/tags address lists the downloaded models; we keep only their names.
        names = [m["name"] for m in requests.get(OLLAMA_URL + "/api/tags", timeout=5).json()["models"]]   # model names
        server_ok = True                                  # the server answered
    except Exception:                                     # no answer: Ollama is not running
        names, server_ok = [], False                      # so no models are visible
    check("Ollama server at " + OLLAMA_URL, server_ok, "open the Ollama app (or `ollama serve`), or set USE_RECORDED = True")   # server
    check("model " + MODEL, MODEL in names, "ollama pull " + MODEL)                       # chat model downloaded?
    check("model " + EMBED_MODEL, EMBED_MODEL in names, "ollama pull nomic-embed-text")   # embedding model downloaded?
print("Mode:", "REPLAY recorded replies" if USE_RECORDED else "LIVE Ollama calls")       # which mode is on

Python 3.11.17 on Windows
OK      Python 3.9 or newer
OK      package pandas (Week 10)
OK      package requests (Week 10)
OK      package sklearn (Week 10)
OK      package matplotlib (Week 10)
OK      package pydantic (Week 10)
OK      package numpy (Week 11)
OK      package chromadb (Week 11)
OK      Ollama server at http://127.0.0.1:11434
OK      model qwen2.5:3b
OK      model nomic-embed-text:latest
Mode: LIVE Ollama calls


In [3]:
# PREPARED — the two Ollama helpers: post_to_ollama() for the chat model (as in Week 10) and
# embed_texts() for the embedding model. Both support replay: in replay mode they return the
# recorded real answer instead of calling Ollama. Every live reply is also saved in outputs/, so
# your own run can be replayed offline later. Result: two functions used by every later cell.
import numpy as np                                        # numpy = numbers, vectors and matrices

# File names. The recorded files come with the lab; the "MY_" files collect your own live results.
# LLM = large language model, i.e. the chat model.
REPLIES_FILE = Path("llm_replies_week11.json")            # recorded real LLM replies (used when USE_RECORDED = True)
EMBEDS_FILE = Path("embeddings_week11.json")              # recorded real vectors, also for replay
Path("outputs").mkdir(exist_ok=True)                      # folder for everything this notebook saves
MY_REPLIES_FILE = Path("outputs/my_llm_replies.json")     # every live LLM reply you get is saved here
MY_EMBEDS_FILE = Path("outputs/my_embeddings.json")       # every live vector you get is saved here
# Load the recorded files into dictionaries (an empty dictionary if a file is missing).
RECORDED = json.loads(REPLIES_FILE.read_text(encoding="utf-8")) if REPLIES_FILE.exists() else {}       # load replies
RECORDED_EMB = json.loads(EMBEDS_FILE.read_text(encoding="utf-8")) if EMBEDS_FILE.exists() else {}    # load vectors
MY_REPLIES, MY_EMBEDS = {}, {}                            # this session's live replies and vectors
USED_KEYS, USED_EMB = set(), set()                        # which requests this run used (for the run record)

# A replay key is a short fingerprint of a request: the same model and messages always give the
# same key, so we can look up the recorded reply. Any change to the prompt text gives a new key.
def reply_key(payload):                                   # replay key of one chat request
    # Join the model name and the text of every message into one string.
    text = payload["model"] + "\n" + "\n".join(m["content"] for m in payload["messages"])   # model + all messages
    # SHA-256 (a hash function) turns the string into a fixed code; we keep its first 16 characters.
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]                            # short fingerprint

# Send ONE chat request to Ollama (or replay it). It returns two things: the reply envelope (a dict
# with the answer text, token counts and timings) and the number of seconds the request took.
def post_to_ollama(payload, timeout=600):                 # send ONE chat request; returns (envelope, seconds)
    key = reply_key(payload)                              # fingerprint of this exact request
    USED_KEYS.add(key)                                    # remember that this run used it
    # Replay mode: look the reply up by its key instead of calling Ollama.
    if USE_RECORDED:                                      # replay mode: look the reply up, no Ollama
        if key not in RECORDED:                           # the prompt differs from the recorded one
            # Stop with an error that explains the problem: the prompt text was changed.
            raise KeyError("no recorded reply for this exact prompt: check it matches the tutorial")   # say why
        return RECORDED[key]["envelope"], RECORDED[key]["seconds"]   # the recorded real reply and its time
    # Live mode, step 1: if this exact request was already sent in this session, reuse that reply.
    if key in MY_REPLIES:                                 # the identical request was already sent in this session
        return MY_REPLIES[key]["envelope"], MY_REPLIES[key]["seconds"]   # reuse the earlier reply
    # Live mode, step 2: send the request to Ollama and measure how long it takes.
    start = time.perf_counter()                           # start the stopwatch
    r = requests.post(OLLAMA_URL + "/api/chat", json=payload, timeout=timeout)   # the real request
    seconds = round(time.perf_counter() - start, 2)       # wall-clock seconds for this request
    r.raise_for_status()                                  # stop on HTTP errors (e.g. model not found)
    envelope = r.json()                                   # Ollama's reply envelope (message, token counts, durations)
    # Save the request and its reply in MY_REPLIES and write the file now, so nothing is lost.
    MY_REPLIES[key] = {"model": payload["model"], "messages": payload["messages"],   # what was sent ...
                       "envelope": envelope, "seconds": seconds}                     # ... and what came back
    MY_REPLIES_FILE.parent.mkdir(exist_ok=True)           # create outputs/ only when a reply is really saved
    MY_REPLIES_FILE.write_text(json.dumps(MY_REPLIES, indent=1, ensure_ascii=False), encoding="utf-8")   # save now
    return envelope, seconds                              # the reply and its time

# The same idea for embeddings: the replay key is a fingerprint of the model name and input text.
def emb_key(model, text):                                 # replay key of one embedding request
    # The key is the first 16 characters of the SHA-256 fingerprint of the model name plus the text.
    return hashlib.sha256((model + "\n" + text).encode("utf-8")).hexdigest()[:16]   # model + input text

# The JSON files store each vector as base64 text (base64 = a way to write raw bytes as letters
# and digits). float32 = a number stored in 4 bytes. These two helpers convert exactly, both ways.
def vec_to_text(v):                                       # turn a float32 vector into base64 text
    return base64.b64encode(np.asarray(v, dtype="<f4").tobytes()).decode("ascii")   # 768 x 4 bytes, no rounding

def text_to_vec(s):                                       # turn base64 text back into the vector
    return np.frombuffer(base64.b64decode(s), dtype="<f4")   # back to the same 768 float32 numbers

# Embed a list of texts and return a matrix with one row (one vector of 768 numbers) per text.
def embed_texts(texts, model=EMBED_MODEL):                # returns a matrix: one row per text
    rows = []                                             # one vector per text
    for text in texts:                                    # one request per text (simple to record and replay)
        key = emb_key(model, text)                        # fingerprint of this exact input
        USED_EMB.add(key)                                 # remember that this run used it
        # Replay mode: take the recorded vector of this text; Ollama is not needed.
        if USE_RECORDED:                                  # replay mode: the recorded vector, no Ollama
            rows.append(text_to_vec(RECORDED_EMB[key]["vector"]))   # KeyError = the text differs from the recording
            continue                                      # go on with the next text
        # Live mode: ask Ollama only for texts that were not embedded yet in this session.
        if key not in MY_EMBEDS:                          # not embedded yet in this session
            start = time.perf_counter()                   # start the stopwatch
            # Send the text to Ollama's /api/embed address, which answers with its vector.
            r = requests.post(OLLAMA_URL + "/api/embed", json={"model": model, "input": text}, timeout=600)   # request
            r.raise_for_status()                          # stop on HTTP errors
            # Keep the vector as float32, the precision that Chroma (our vector database) stores.
            vector = np.asarray(r.json()["embeddings"][0], dtype="<f4")   # float32, the precision Chroma stores
            MY_EMBEDS[key] = {"model": model, "input": text, "vector": vec_to_text(vector),   # what was sent + vector
                              "seconds": round(time.perf_counter() - start, 3)}              # and its time
        rows.append(text_to_vec(MY_EMBEDS[key]["vector"]))   # the same exact numbers that the file stores
    # Live mode: save all vectors so far to outputs/my_embeddings.json (one save per call).
    if not USE_RECORDED:                                  # live mode: save all vectors so far
        MY_EMBEDS_FILE.parent.mkdir(exist_ok=True)        # create outputs/ only when vectors are really saved
        MY_EMBEDS_FILE.write_text(json.dumps(MY_EMBEDS, ensure_ascii=False), encoding="utf-8")   # one save per call
    return np.vstack(rows)                                # a (number of texts) x 768 matrix

# Add up the recorded seconds of the requests whose keys are given (requests missing from the store
# are skipped). It gives the same number in live and in replay mode.
def recorded_seconds(keys, store):                        # total recorded seconds of some requests
    return round(sum(store[k]["seconds"] for k in keys if k in store), 2)   # same number live and in replay

In [4]:
# Exercise 2 — one chat request and one embedding request, to see what comes back. PREPARED.
# The chat model returns text; the embedding model returns a list of numbers (a vector) for the same kind of text.
# Result: the printed answer and its time, then the vector's length, first 5 numbers and norm.

# Build the chat request (payload = the data we send): the model, one question and fixed settings.
payload = {"model": MODEL, "stream": False,                          # one complete reply, not a stream
           "messages": [{"role": "user", "content":                  # one user message, no chat history
                         "Which Python library gives us DataFrames? Answer with one word."}],   # the question
           # temperature 0 = least random; seed = fixed starting point;
           # num_predict = stop after at most 20 tokens (a token = a word or part of a word).
           "options": {"temperature": 0, "seed": SEED, "num_predict": 20}}   # same settings as Week 10
envelope, seconds = post_to_ollama(payload)                          # send it (or replay it)
print("chat answer    :", envelope["message"]["content"], f"({seconds} s)")   # the model's text and the time

# Now one embedding request: embed_texts() takes a list of texts; [0] takes the first row (vector).
vec = embed_texts(["How do I open the MLflow dashboard to compare runs?"])[0]   # one embedding vector
print("vector length  :", len(vec))                                  # nomic-embed-text gives 768 numbers
print("first 5 numbers:", np.round(vec[:5], 4))                      # a peek: the numbers alone mean nothing to us
# The norm is the length of the vector. Ollama returns vectors of length 1 (unit-length vectors).
print("vector length (norm):", round(float(np.linalg.norm(vec)), 4)) # Ollama returns unit-length vectors (norm 1)

chat answer    : pandas (0.24 s)
vector length  : 768
first 5 numbers: [-0.004   0.0008 -0.1695 -0.0464  0.0584]
vector length (norm): 1.0


In [57]:
# Exercise 3.1 — load the six tutorials into a table (a pandas DataFrame), one row per file.
# The week number comes from the FILE NAME (trustworthy application metadata), never from the LLM.
# Metadata = facts about a text (its week, its file name) stored next to the text.
# Result: docs_df, a table with one row per tutorial and its size in characters and words.
import pandas as pd                                       # pandas = tables (DataFrames)
from pydantic import BaseModel, ConfigDict, Field         # data contracts, as in Week 10

CORPUS = Path("../data/")                                   # the folder with the six .md tutorials

# A data contract (a pydantic model): every document must have exactly these four fields with these
# types. If a value is wrong (e.g. week 7), pydantic stops with an error, so no bad data is kept.
class DocRecord(BaseModel):                               # the contract for one loaded document
    model_config = ConfigDict(extra="forbid")             # no extra fields allowed
    doc_id: str                                           # the file name without the .md ending
    doc: int = Field(ge=1, le=3)                         # 1 to 6: ge = at least, le = at most
    source_file: str                                      # the file name, used in citations
    text: str                                             # the full raw text

# Read every weekNN_*.md file (NN = the two-digit week number) and return one table row per file.
def load_docs(folder=CORPUS):                             # read every weekNN_*.md file in the folder
    rows = []
                                          # one DocRecord per file
    for path in sorted(folder.glob("2017-43-p*.txt")):          # week01 to week06, in sorted order
        # Build and check one record. path.name[4:6] takes characters 4 and 5 of the file name,
        # so "week04_..." gives "04", and int() turns that into the week number 4.  
        rec = DocRecord(doc_id=path.stem, doc=int(path.name[9]), source_file=path.name,   # "week04_..." -> 4
                        text=path.read_bytes().decode("utf-8"))   # raw text, line endings untouched
        rows.append(rec.model_dump())                     # the checked record, as a plain dict
    return pd.DataFrame(rows)                             # turn the list of dicts into a table

# Run it, then add two size columns so we can see how big each tutorial is.
docs_df = load_docs()                                     # docs_df: one row per tutorial file
docs_df["chars"] = docs_df["text"].str.len()              # number of characters in each file
docs_df["words"] = docs_df["text"].str.split().str.len()  # whitespace-separated words (code included)
print(docs_df[["doc", "source_file", "chars", "words"]].to_string(index=False))   # show the table
print("documents:", len(docs_df), "| total words:", docs_df["words"].sum())        # 6 files expected

 doc    source_file  chars  words
   1 2017-43-p1.txt 107025  18495
   2 2017-43-p2.txt 187536  33048
   3 2017-43-p3.txt 200393  34364
documents: 3 | total words: 85907


In [58]:
# Exercise 3.2 — clean the text without losing evidence: one kind of line ending,
# no spaces or tabs at the end of lines, and at most one empty line in a row. Headings and code
# blocks stay, because answers often live in code (e.g. fillna(-1)). Result: docs_df["clean"].
# regex = a text pattern; re.sub(pattern, new, text) replaces every match of the pattern with new.
def clean_text(text):                                     # takes raw text, returns clean text
    # Three cleaning steps, in order; each step changes `text` and passes it on to the next.
    text = text.replace("\r\n", "\n")                     # turn Windows line endings into \n
    text = re.sub(r"[ \t]+\n", "\n", text)                # remove spaces and tabs at the end of lines
    return re.sub(r"\n{3,}", "\n\n", text)                # 3 or more line breaks become 2

# Clean all six documents and compare the length before (chars) and after (clean_chars).
docs_df["clean"] = docs_df["text"].map(clean_text)        # apply clean_text() to every document
docs_df["clean_chars"] = docs_df["clean"].str.len()       # length after cleaning
print(docs_df[["source_file", "chars", "clean_chars"]].to_string(index=False))   # how much was removed

# Safety check: each gold quote (the exact tutorial text that answers a gold question) must still
# be found word for word in the cleaned text; otherwise cleaning would have destroyed evidence.
# gold_q = pd.read_csv("gold/gold_questions.csv")           # the 11 gold questions (Q11 has no answer in the corpus)
# for q in gold_q[gold_q["answerable"]].itertuples():       # only Q01 to Q10, which have an answer
#     text = docs_df.loc[docs_df["week"] == q.week, "clean"].iloc[0]   # the cleaned tutorial of that week
#     print(q.id, "quote still in the cleaned text:", q.quote in text) # True expected for all ten

   source_file  chars  clean_chars
2017-43-p1.txt 107025       105409
2017-43-p2.txt 187536       185238
2017-43-p3.txt 200393       196399


In [63]:

def split_sections(text):                                 # returns a list of (heading, text) pairs
    # Start: no sections yet, heading "(top)" for any text before the first heading.
    sections, heading, lines = [], "(top)", []            # results, current heading, its lines
    for line in text.split("\n"):                         # walk through the document line by line
        # Match "Part X" or "Division X.Y" at the start of a line based on image_ff116a.png[cite: 4]
        # m = re.match(r"^(Part\s+\d+|Division\s+\d+\.\d+)", line)
        m = re.match(r"^(Part\s+\d+)", line)
        # a new "Part X" starts a new section
        # Might produce more than 21 parts, because there are multiple mentions of "Part X" in table of contents, etc
        if m:                                             
            if "".join(lines).strip():                    # save the previous section if it has text
                sections.append((heading, "\n".join(lines))) 
            # Start the new section: its heading is the full line text.
            heading, lines = line.strip(), [line]         
        else:                                             # an ordinary line or numbered item
            lines.append(line)                            # add it to the current section
            
    # After the loop, save the last section of the file too.
    if "".join(lines).strip():                            # if the last section has text
        sections.append((heading, "\n".join(lines)))      # save it too
    return sections                                       # the list of (heading, text) pairs

# Run split_sections() on every document and build one table with one row per section.
def make_sections(docs):                                  # returns the sections table
    rows = []                                             # one row per section
    for d in docs.itertuples():                           # go through every document
        for i, (heading, body) in enumerate(split_sections(d.clean)):   # every section, numbered from 0
            # A stable id, e.g. w04_s005 = week 4, section 5 (:02d writes the week with 2 digits).
            rows.append({"section_id": f"doc{d.doc}_s{i:02d}", "doc": d.doc,  # stable id: week + section number
                         "source_file": d.source_file, "section": heading, "text": body})   # where it is + its text
    return pd.DataFrame(rows)                             # the sections table

# Run it and look at the result: how many sections there are, and how long the longest one is.
sections_df = make_sections(docs_df)                      # run it on the six cleaned tutorials
print("sections:", len(sections_df), "| per doc:", sections_df.groupby("doc").size().tolist())   # 125 expected
# Some sections are long, which is why Exercise 4.2 cuts long sections into smaller windows.
print("longest section:", sections_df["text"].str.len().max(), "characters")                      # why we need windows
# Show the first 8 section ids and headings of doc 3
print(sections_df[sections_df["doc"] == 3][["section_id", "section"]].head(8).to_string(index=False))   # a peek

sections: 57 | per doc: [29, 7, 21]
longest section: 68221 characters
section_id                                                                                 section
  doc3_s00                                            Part 15\tAdditional rules for bicycle riders
  doc3_s01                                  Part 16\tRules for people travelling in or on vehicles
  doc3_s02 Part 17\tAdditional rules for drivers of trams, tram recovery vehicles and public buses
  doc3_s03                                                       Part 18\tMiscellaneous road rules
  doc3_s04                                                                     Part 19\tExemptions
  doc3_s05                              Part 20\tTraffic control devices and traffic-related items
  doc3_s06                                                                        Part 21\tGeneral
  doc3_s07                                                                                  Part 4
